# Lubay - Synthetic Data Generation Using RAGAS

> Generate synthetic test data for evaluating the Lubay mental health RAG system.

This notebook will:

1. Load the stress management PDF guide
2. Use RAGAS to generate synthetic test questions
3. Create a LangSmith dataset for evaluation
4. Provide test data for tuning chunking, prompts, and topK

The synthetic data will help you test Lubay with realistic questions about stress management and mental health.



## Runtime (`MODE`)

Change **one variable** in the next cell, then run the setup cells.

| `MODE` | Where you run | LLM API |
| --- | --- | --- |
| `"vocareum"` | Local (`uv sync`, kernel `.venv/bin/python`) | [Vocareum](https://help.vocareum.com/en/articles/9341351-genai-in-vocareum-labs) gateway |
| `"openai"` | Local (`uv sync`, kernel `.venv/bin/python`) | Official OpenAI API |
| `"colab"` | [Google Colab](https://colab.research.google.com/) | Vocareum gateway (pip + upload `data/`) |

LangSmith is required in every mode.



In [1]:
from typing import Literal

from pydantic import BaseModel, Field

# Change this:
MODE = "vocareum"  # "vocareum" | "openai" | "colab"


class SdgSettings(BaseModel):
    """Notebook runtime: where we run, and which OpenAI-compatible API we call."""

    mode: Literal["vocareum", "openai", "colab"] = Field(
        description="vocareum / openai = local uv; colab = Google Colab + Vocareum."
    )

    @property
    def is_colab(self) -> bool:
        return self.mode == "colab"

    @property
    def use_vocareum(self) -> bool:
        return self.mode in ("vocareum", "colab")

    @property
    def openai_base_url(self) -> str | None:
        if self.use_vocareum:
            return "https://openai.vocareum.com/v1"
        return None

    @property
    def generator_model(self) -> str:
        return "gpt-4o-mini" if self.use_vocareum else "gpt-4o-mini"

    @property
    def rag_model(self) -> str:
        return "gpt-4o-mini" if self.use_vocareum else "gpt-4o-mini"

    @property
    def eval_model(self) -> str:
        return "gpt-4o-mini" if self.use_vocareum else "gpt-4o-mini"

    embed_small: str = "text-embedding-3-small"
    embed_large: str = "text-embedding-3-large"

    @property
    def llm_key_prompt(self) -> str:
        return "Vocareum API key: " if self.use_vocareum else "OpenAI API key: "


cfg = SdgSettings(mode=MODE)
print(f"MODE = {cfg.mode}")
print(f"API  = {'Vocareum ' + cfg.openai_base_url if cfg.openai_base_url else 'OpenAI (default base URL)'}")
print(f"chat = generator {cfg.generator_model} / rag {cfg.rag_model} / eval {cfg.eval_model}")



MODE = vocareum
API  = Vocareum https://openai.vocareum.com/v1
chat = generator gpt-4o-mini / rag gpt-4o-mini / eval gpt-4o-mini


## Setup

**Local (`vocareum` or `openai`):** `uv sync`, select kernel **`.venv/bin/python`**, run the next two cells.

**Colab (`colab`):** run the install cell once, **Runtime → Restart session**, re-run the `MODE` cell, skip install, then run the project-files cell.

Do not `pip install` into Homebrew Python — that triggers `externally-managed-environment`.



In [2]:
import sys
import subprocess
from pathlib import Path

print(f"Python: {sys.executable}")
print(f"MODE:   {cfg.mode}")

if cfg.is_colab:
    if "google.colab" not in sys.modules:
        raise SystemExit(
            "MODE is 'colab' but this is not Colab. "
            "Set MODE to 'vocareum' or 'openai' for local uv."
        )
    try:
        import langchain_openai  # noqa: F401
        import ragas  # noqa: F401
        print("Packages already present. Continue to project files.")
    except ImportError:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-qU",
                "pydantic",
                "langsmith>=0.7.24",
                "ragas",
                "datasets",
                "langchain",
                "langchain-core",
                "langchain-classic",
                "langchain-text-splitters",
                "langchain-qdrant",
                "langchain-openai",
                "langchain-community",
                "pymupdf",
                "nest-asyncio",
                "unstructured",
                "nltk",
            ]
        )
        print("Installed. Runtime → Restart session, re-run MODE, then the project-files cell.")
else:
    if "google.colab" in sys.modules:
        raise SystemExit("This is Colab. Set MODE = 'colab' at the top, then re-run from there.")
    if ".venv" not in sys.executable:
        raise SystemExit(
            "Wrong kernel. Select `.venv/bin/python` "
            "(Command Palette → Notebook: Select Notebook Kernel)."
        )
    print("Local uv environment looks good. Continue to project files.")



Python: /Users/klengramos/14A-nextjs-rag/.venv/bin/python
MODE:   vocareum
Local uv environment looks good. Continue to project files.


### Project files

Local modes expect `data/` at the project root containing the stress management PDF.



In [3]:
from pathlib import Path
import os


def _data_ready(root: Path) -> bool:
    data = root / "data"
    return data.is_dir() and any(data.glob("*.pdf"))


def _find_root() -> Path:
    for candidate in [Path("."), *Path(".").glob("*/")]:
        if _data_ready(candidate):
            return candidate
    return Path(".")


root = _find_root()

os.chdir(root)

if not _data_ready(Path(".")):
    raise SystemExit(
        "Run from the project root so `data/` is on the path "
        "with the stress management PDF."
    )

print("cwd:", Path.cwd())
print("Project files are in place.")



cwd: /Users/klengramos/14A-nextjs-rag
Project files are in place.


## Task 1: Dependencies and API Keys

We'll need a number of API keys and dependencies for this pipeline:

1. An LLM key — Vocareum when `MODE` is `"vocareum"` or `"colab"`, or an OpenAI key when `MODE` is `"openai"`
2. QDrant as our vectorstore
3. LangSmith for our evaluation coordinator!

Let's provide all the required information below.



## API keys

Set LangSmith and the LLM key in the cells below (or export them in the shell before launching Jupyter).



### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.



In [4]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')



[nltk_data] Downloading package punkt to
[nltk_data]     /Users/klengramos/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/klengramos/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [5]:
import os
import getpass
api_key = os.environ.get("LANGCHAIN_API_KEY") or getpass.getpass("LangChain API Key:")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = api_key



We'll also want to set a project name to make things easier for ourselves.



In [6]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"Lubay - SDG - {uuid4().hex[0:8]}"



The next cell uses `cfg` from `MODE`: Vocareum sets `OPENAI_BASE_URL` to `https://openai.vocareum.com/v1`; OpenAI mode uses the default OpenAI endpoint.



In [7]:
import os
from getpass import getpass

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

api_key = os.environ.get("OPENAI_API_KEY") or getpass(cfg.llm_key_prompt)
os.environ["OPENAI_API_KEY"] = api_key

if cfg.openai_base_url:
    os.environ["OPENAI_BASE_URL"] = os.environ.get("OPENAI_BASE_URL") or cfg.openai_base_url
    os.environ["OPENAI_API_BASE"] = os.environ["OPENAI_BASE_URL"]
else:
    os.environ.pop("OPENAI_BASE_URL", None)
    os.environ.pop("OPENAI_API_BASE", None)


def make_chat(model: str, **kwargs) -> ChatOpenAI:
    params = {"model": model, "api_key": os.environ["OPENAI_API_KEY"], **kwargs}
    if cfg.openai_base_url:
        params["base_url"] = cfg.openai_base_url
    return ChatOpenAI(**params)


def make_embeddings(model: str, **kwargs) -> OpenAIEmbeddings:
    params = {"model": model, "api_key": os.environ["OPENAI_API_KEY"], **kwargs}
    if cfg.openai_base_url:
        params["base_url"] = cfg.openai_base_url
    return OpenAIEmbeddings(**params)


print("LLM client ready:", "Vocareum" if cfg.use_vocareum else "OpenAI")



LLM client ready: Vocareum


## Generating Synthetic Test Data

We will be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll load the stress management PDF guide from the Mental Health Foundation.


In [8]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents from {path}")


Loaded 26 documents from data/


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

In [9]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

generator_llm = LangchainLLMWrapper(make_chat(cfg.generator_model))
generator_embeddings = LangchainEmbeddingsWrapper(make_embeddings(cfg.embed_small))



/Users/klengramos/14A-nextjs-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/3n/p6hvgwm54m33y_w0p3fkmd_h0000gn/T/ipykernel_3775/3010198232.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(make_chat(cfg.generator_model))
/var/folders/3n/p6hvgwm54m33y_w0p3fkmd_h0000gn/T/ipykernel_3775/3010198232.py:5: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddi

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [10]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg


KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [11]:
from ragas.testset.graph import Node, NodeType

# Load all documents into the knowledge graph
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
print(f"Added {len(kg.nodes)} nodes to knowledge graph")
kg


Added 26 nodes to knowledge graph


KnowledgeGraph(nodes: 26, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [12]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
print(f"Applied transforms. Knowledge graph now has {len(kg.relationships)} relationships")
kg


Applying CustomNodeFilter:   0%|          | 0/26 [00:00<?, ?it/s]Node bd22d399-5111-4457-9039-575a80cd76cd does not have a summary. Skipping filtering.
Node 9cf1b127-900c-4847-9937-2411a44e21fa does not have a summary. Skipping filtering.
Node 26bb8868-62c3-4daf-91bd-4d903e8b6241 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:  15%|█▌        | 4/26 [00:01<00:05,  3.86it/s]Node 204306d1-f4cd-41fc-a488-7928778fed7a does not have a summary. Skipping filtering.
Node 907f55f5-b4cf-4798-a542-e03a1440e964 does not have a summary. Skipping filtering.
Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 95.86it/s]

Applied transforms. Knowledge graph now has 80 relationships


KnowledgeGraph(nodes: 26, relationships: 80)

We can save and load our knowledge graphs as follows.

In [13]:
kg.save("data/lubay_knowledge_graph.json")
lubay_data_kg = KnowledgeGraph.load("data/lubay_knowledge_graph.json")
print(f"Loaded knowledge graph with {len(lubay_data_kg.nodes)} nodes")
lubay_data_kg


Loaded knowledge graph with 26 nodes


KnowledgeGraph(nodes: 26, relationships: 80)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=lubay_data_kg)


However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthesizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [15]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]


### Generate Test Set

Finally, we can use our `TestSetGenerator` to generate our testset!


In [16]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()


Generating Samples: 100%|██████████| 11/11 [00:02<00:00,  3.93it/s]


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,How does strss impact mental helth and what ar...,[3\nWhat is stress?\nStress affects us in a nu...,Stress affects us both physically and emotiona...,Corporate Wellness Coordinator,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,What indigestion mean when stress happen?,[4\nPhysical symptoms \nPeople react different...,"Indigestion can occur as a result of stress, w...",Health and Wellness Coach,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
2,Wat are the common signs of stres that people ...,[7\nIdentifying the signs of stress\nEveryone ...,Common signs and symptoms of stress include fe...,Health and Wellness Coach,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
3,How can healthy relationships contribute to ma...,[9\nRelationships\nRelationships are a great s...,Healthy relationships serve as a great support...,Corporate Wellness Coordinator,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
4,What were the impacts of mental health on work...,[10\nWork-life balance \nThe pressure of an in...,"In 2008, mental health accounted for 442,000 c...",Corporate Wellness Coordinator,MISSPELLED,LONG,single_hop_specific_query_synthesizer
5,Wot are the ways to seek help for debt problem...,[<1-hop>\n\n17\nStepChange\nStepChange provide...,"To seek help for debt problems, individuals ca...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,How does the body's stress response relate to ...,[<1-hop>\n\n13\nHow can you help yourself?\nSt...,The body's stress response is a natural reacti...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,Wht are the signs of stress and how can you ma...,[<1-hop>\n\n18\n101 Top tips from you\nThank y...,The signs of stress include feelings of consta...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,How can Citizens Advice help individuals deali...,[<1-hop>\n\n11\nMoney\nMoney and debt concerns...,Citizens Advice can help individuals dealing w...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,What role does a GP play in addressing the men...,[<1-hop>\n\n11\nMoney\nMoney and debt concerns...,A GP plays a crucial role in addressing the me...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [17]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)


Applying SummaryExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/26 [00:00<?, ?it/s]Node 6aba0017-0b9c-4b0c-bed8-b55ad7538a73 does not have a summary. Skipping filtering.
Node e032cda1-5140-424e-8190-d4bff4308c0f does not have a summary. Skipping filtering.
Node 2b5ccd33-3307-4b1c-ac85-28f865268cf4 does not have a summary. Skipping filtering.
Generating Samples: 100%|██████████| 12/12 [02:06<00:00, 10.52s/it]


In [18]:
dataset.to_pandas()


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What stress do to us?,[3\nWhat is stress?\nStress affects us in a nu...,"Stress affects us in a number of ways, both ph...",Health and Wellness Coach,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,What kinda things can make you feel dizzy when...,[4\nPhysical symptoms \nPeople react different...,"When you are stressed out, you may experience ...",Health and Wellness Coach,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
2,What are the common signs of stress that can i...,[7\nIdentifying the signs of stress\nEveryone ...,Common signs and symptoms of stress that can a...,Health and Wellness Coach,MISSPELLED,LONG,single_hop_specific_query_synthesizer
3,How can relationships impact stress levels?,[9\nRelationships\nRelationships are a great s...,Relationships can provide great support during...,Health and Wellness Coach,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
4,How money problems make stress response worse ...,[<1-hop>\n\n11\nMoney\nMoney and debt concerns...,Money and debt concerns place huge pressure on...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,How can crisis intervention services assist in...,[<1-hop>\n\n17\nStepChange\nStepChange provide...,"Crisis intervention services, such as those pr...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,How can spending time with children contribute...,[<1-hop>\n\n21\n65.\t Spend time with children...,Spending time with children can contribute to ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,How can going for a bike ride and engaging in ...,[<1-hop>\n\n22\n85.\t Go for a bike ride\n86.\...,Going for a bike ride is a form of physical ex...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,What are the connections between prolonged str...,[<1-hop>\n\n3\nWhat is stress?\nStress affects...,Prolonged stress can lead to mental health pro...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,How does the work-life balance in the UK contr...,[<1-hop>\n\n25\nThe Mental Health Foundation \...,The work-life balance in the UK significantly ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [36]:
from langsmith import Client

client = Client()

dataset_name = "Lubay Stress Management"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Stress management and mental health questions for Lubay"
)


We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [37]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )
print(f"Created {len(dataset.to_pandas())} examples in LangSmith dataset")


Created 12 examples in LangSmith dataset


## Basic RAG Chain

Time for some RAG!


In [109]:
rag_documents = docs


To keep things simple, we'll just use LangChain's recursive character text splitter!


In [110]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 120, #110
    chunk_overlap = 60 #40
)

rag_documents = text_splitter.split_documents(rag_documents)


We'll create our vectorstore using `cfg.embed_small` (`text-embedding-3-small` via Vocareum or OpenAI, depending on `MODE`).



In [111]:
embeddings = make_embeddings(cfg.embed_small)



As usual, we will power our RAG application with Qdrant!

In [112]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Lubay RAG"
)


In [113]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})


To get the "A" in RAG, we'll provide a prompt.

In [114]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
'You are Lubay, a compassionate mental health assistant created by the Mental Health Foundation. ' +
      'You specialize in stress management, anxiety, coping strategies, and mental wellness. ' +
      'Use the getInformation tool to find relevant information from the Mental Health Foundation guide provided as a source. ' +
      'Always provide evidence-based responses. If the source document does not cover something, ' +
      'acknowledge the limitation that the information provided comes from mental health foundation UK but the guide is applicable to anyone outside UK.
      and suggest consulting a healthcare professional that is non-imposive and supportive and say that the provided in the guide is a UK number but you can help 
      in searching for local government numbers.'+
      'Make sure you only provide answers within the provided corpus and provide related responses that might be help' +
      'Be non-judgmental in all responses.".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)


For our LLM, we will be using the model configured in `cfg.rag_model`.



In [115]:
llm = make_chat(cfg.rag_model)



Finally, we can set-up our RAG LCEL chain!

In [116]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)


In [117]:
rag_chain.invoke({"question" : "What are the common signs of stress?"})


"Common signs of stress can manifest in various physical, emotional, and behavioral ways. According to the Mental Health Foundation, some of the signs and symptoms to look out for include:\n\n- **Feelings of constant worry or anxiety**\n- **Irritability or mood swings**\n- **Difficulty relaxing**\n- **Depression**\n- **Low self-esteem**\n- **Changes in appetite (eating more or less than usual)**\n- **Sleeping Problems (insomnia or oversleeping)**\n\nIf you notice these signs in yourself or someone else, it's important to acknowledge them and consider seeking support. Stress affects everyone differently, and recognizing these symptoms is the first step toward managing it effectively. If you feel the need for more personalized assistance, please consider consulting a healthcare professional for supportive guidance. Remember, while the resources provided are relevant in the UK, they can be useful for anyone outside the UK as well. If you need help finding local mental health resources or 

## LangSmith Evaluation Set-up

We'll use `cfg.eval_model` as our evaluation LLM for our base Evaluators.



In [118]:
eval_llm = make_chat(cfg.eval_model, temperature=0)



We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [119]:
from typing import Optional
 
from pydantic import BaseModel, Field
 
from langsmith.evaluation import evaluate
from langsmith.schemas import Example, Run
from langchain_classic.evaluation import load_evaluator
 
 
class EvaluatorInputs(BaseModel):
    """Payload the string graders expect: chain output vs dataset row."""
 
    prediction: str = Field(description="RAG chain answer (run.outputs['output'])")
    reference: Optional[str] = Field(
        default=None, description="Gold answer from the dataset"
    )
    input: Optional[str] = Field(
        default=None, description="Question from the dataset"
    )
 
 
def prepare_eval_data(run: Run, example: Optional[Example] = None) -> dict:
    outputs = run.outputs or {}
    example_outputs = (example.outputs if example else None) or {}
    example_inputs = (example.inputs if example else None) or {}
    prediction = outputs.get("output")
    if prediction is None and len(outputs) == 1:
        prediction = next(iter(outputs.values()))
    return EvaluatorInputs(
        prediction=str(prediction or ""),
        reference=example_outputs.get("answer"),
        input=example_inputs.get("question"),
    ).model_dump()
 
 
def make_langchain_string_evaluator(
    evaluator_type, config=None, prepare_data=None, result_key=None
):
    """Create a LangChain string evaluator compatible with langsmith evaluate()."""
    eval_config = config or {}
    llm = eval_config.pop("llm", None)
    criteria = eval_config.pop("criteria", None)
    eval_kwargs = {}
    if llm is not None:
        eval_kwargs["llm"] = llm
    if criteria is not None:
        eval_kwargs["criteria"] = criteria
    evaluator = load_evaluator(evaluator_type, **eval_kwargs)
 
    # FIX 1: key off the actual criterion name (when there's exactly one) instead
    # of the loader type string, so two "criteria" or "labeled_criteria" evaluators
    # don't both report under the same LangSmith feedback key and overwrite each other.
    key = result_key or (
        next(iter(criteria))
        if isinstance(criteria, dict) and len(criteria) == 1
        else evaluator_type
    )
 
    def _prepare_and_evaluate(run, example=None):
        data = prepare_data(run, example) if prepare_data else {}
        eval_call_kwargs = {
            "prediction": data.get("prediction", ""),
            "input": data.get("input", ""),
        }
        # FIX 3: only pass reference to evaluators that actually use it (qa,
        # labeled_criteria, etc.) — passing it to an unlabeled CriteriaEvalChain
        # triggers "Ignoring reference in CriteriaEvalChain" every call.
        if getattr(evaluator, "requires_reference", False):
            eval_call_kwargs["reference"] = data.get("reference", "")
        result = evaluator.evaluate_strings(**eval_call_kwargs)
        # FIX 2: LangSmith's EvaluationResult has no "reasoning" field and forbids
        # extra fields, so passing it through raises a ValidationError and kills
        # the evaluator run. Map it to "comment" instead.
        if "reasoning" in result:
            result["comment"] = result.pop("reasoning")
        return {"key": key, **result}
 
    return _prepare_and_evaluate
 
 
# Make sure this is defined before the calls below, e.g.:
# from langchain_openai import ChatOpenAI
# eval_llm = ChatOpenAI(model="gpt-4o", temperature=0)
 
qa_evaluator = make_langchain_string_evaluator(
    "qa",
    config={"llm": eval_llm},
    prepare_data=prepare_eval_data,
)
 
labeled_helpfulness_evaluator = make_langchain_string_evaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm": eval_llm,
    },
    prepare_data=prepare_eval_data,
)
 
empathy_evaluator = make_langchain_string_evaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": (
                "Is this response empathetic? Does it make the user feel"
                " like they are being heard?"
            )
        },
        "llm": eval_llm,
    },
    prepare_data=prepare_eval_data,
)
 


## LangSmith Evaluation

In [120]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator,
    ],
    metadata={"revision_id": "default_chain_init"},
)


View the evaluation results for experiment: 'glossy-energy-26' at:
https://smith.langchain.com/o/6ae57267-2c4e-42c9-bfb3-2080b5a1ea0b/datasets/0e4493fb-c847-48d2-a226-be240a67f006/compare?selectedSessions=d282811d-4a60-45a7-b842-088938d12475




12it [02:13, 11.14s/it]


,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,how can i contact the mentalhealth foundation ...,You can contact the Mental Health Foundation t...,None,You can contact the mentalhealth foundation to...,1,1,0,3.195905,70f29164-be17-4be1-92d4-152b43844c16,01a0e0e5-0530-7092-8fb8-ba77669140f7
1,How can Citizens Advice help individuals manag...,"Citizens Advice provides free, independent, an...",None,"Citizens Advice provides free, independent, an...",1,1,1,3.036246,4e8c9d97-20ab-43b0-8b96-86c9fdff2e58,01a0e0e5-2e40-7e82-b412-8fb6942a4696
2,How does the work-life balance in the UK contr...,The Mental Health Foundation highlights that t...,None,The work-life balance in the UK significantly ...,1,1,1,3.230849,f6a106a8-7040-4b14-815e-285f03a69241,01a0e0e5-5771-7771-b694-cfeaf25dfc1e
3,What are the connections between prolonged str...,Prolonged stress has a significant connection ...,None,Prolonged stress can lead to mental health pro...,1,1,1,4.269615,085fe990-ac09-4f04-832e-91e3d046ef64,01a0e0e5-7f3a-78c2-8b37-c2c680b76f12
4,How can going for a bike ride and engaging in ...,"Engaging in physical exercise, such as going f...",None,Going for a bike ride is a form of physical ex...,1,1,1,3.474320,54125eac-5709-4ca1-8cb4-0b860d498127,01a0e0e5-ad6f-7f23-ab2f-3e4f4c4c46fa
5,How can spending time with children contribute...,While the specific guide may not provide detai...,None,Spending time with children can contribute to ...,1,0,1,3.714617,1940c7fb-033e-438b-af4e-b05cfd8a6a62,01a0e0e5-db07-75f3-bb49-a59c136e230e
6,How can crisis intervention services assist in...,Crisis intervention services can provide essen...,None,"Crisis intervention services, such as those pr...",1,1,1,4.219868,2f1197ac-9aa1-4926-99c2-c99875aff919,01a0e0e6-0592-7d53-9c1b-20a0079bc5ff
7,How money problems make stress response worse ...,Financial issues can significantly exacerbate ...,None,Money and debt concerns place huge pressure on...,1,1,1,3.580701,788f8321-4374-494b-8ccf-6530c53fb9f7,01a0e0e6-3507-76b0-93f5-06c60f9ba993
8,How can relationships impact stress levels?,Relationships can significantly impact stress ...,None,Relationships can provide great support during...,1,1,1,3.538536,42b9f329-2769-43ff-8ae9-307dcf0d49e8,01a0e0e6-5e85-7f73-8835-eb1810f531e3
9,What are the common signs of stress that can i...,Common signs of stress that can impact your we...,None,Common signs and symptoms of stress that can a...,0,0,1,2.910981,e332bb8a-a8bf-47b4-981e-c0a341ac04dc,01a0e0e6-8c87-70d2-b18e-750833837d8e


## Summary

You have successfully:

1. Loaded the stress management PDF guide
2. Generated synthetic test questions using RAGAS
3. Created a LangSmith dataset for evaluation
4. Built a basic RAG chain and evaluated it
5. Built an empathy-enhanced RAG chain and evaluated it

View your evaluation results in LangSmith to compare the two chains!

